# Task 2: Medical Fine-tuning with QLoRA via Unsloth

**Arch Technologies — Generative AI Internship (Month 1)**

This Colab notebook follows Unsloth’s QLoRA workflow (and the [DeepSeek-R1 medical fine-tune walkthrough](https://youtu.be/qcNmOItRw4U)) to adapt a reasoning LLM to clinical Q&A:

1. Load **DeepSeek-R1-Distill-Llama-8B** (or Llama 3.2 3B) in **4-bit**
2. Pull a medical reasoning dataset (`FreedomIntelligence/medical-o1-reasoning-SFT`)
3. Tokenize + format instruction / chain-of-thought / answer triples
4. Attach **LoRA** adapters with Unsloth (`get_peft_model`)
5. Train for **1 epoch** (capped at 60 steps so a free T4 can finish)
6. Report **VRAM**, save the adapter, and test new medical queries

### How to run
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**

This is a learning demo, not a clinical device. Do not use the outputs for real medical decisions.

References: [Unsloth notebooks](https://github.com/unslothai/notebooks) · [DataCamp DeepSeek-R1 fine-tune](https://www.datacamp.com/tutorial/fine-tuning-deepseek-r1-reasoning-model) · [medical-o1-reasoning-SFT](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT)


## 1. GPU check

Unsloth 4-bit training needs a CUDA GPU. Enable a T4 and re-run if this cell fails.


In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime → Change runtime type → T4 GPU, then Runtime → Run all."
)
props = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {props.total_memory / 1024**3:.2f} GB")
print(f"CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")



## 2. Install Unsloth

Same Colab recipe as Task 3 so the stack stays consistent (`unsloth` + `trl==0.22.2`).


In [ ]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install -q unsloth
else:
    import torch
    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.10": "0.0.34", "2.9": "0.0.33.post1", "2.8": "0.0.32.post2"}.get(v, "0.0.34")
    !pip install -q sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install -q --no-deps --upgrade "torchao>=0.16.0"
    !pip install -q transformers==4.56.2
    !pip install -q --no-deps trl==0.22.2



## 3. Configuration

- **4-bit quantization** compresses weights so an 8B model fits on a free T4.
- **LoRA / QLoRA** trains small rank-decomposition matrices instead of every parameter.
- Set `USE_SMALLER_BASE = True` if you hit CUDA out-of-memory (switches to Llama 3.2 3B Instruct).

Optional Hugging Face / W&B tokens are **not** required. Public Unsloth checkpoints and the medical SFT set download without a token.


In [ ]:
from pathlib import Path

USE_SMALLER_BASE = False  # True → Llama 3.2 3B if the 8B run OOMs
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 16
N_SAMPLES = 500
NUM_TRAIN_EPOCHS = 1
MAX_STEPS = 60  # cap so Colab finishes; set to -1 for a full epoch
PER_DEVICE_BATCH = 2
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
ADAPTER_DIR = Path("medical_qlora_adapter")

DEEPSEEK = "unsloth/DeepSeek-R1-Distill-Llama-8B"
LLAMA32 = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MODEL_NAME = LLAMA32 if USE_SMALLER_BASE else DEEPSEEK

print("Base model:", MODEL_NAME)
print(f"QLoRA rank={LORA_R}  alpha={LORA_ALPHA}  4-bit={LOAD_IN_4BIT}")
print(f"Train: {N_SAMPLES} samples, {NUM_TRAIN_EPOCHS} epoch, max_steps={MAX_STEPS}")



## 4. Load 4-bit base model + tokenizer

`FastLanguageModel.from_pretrained(..., load_in_4bit=True)` is Unsloth’s QLoRA entry point.


In [ ]:
from unsloth import FastLanguageModel
import torch

def vram(label: str) -> None:
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"{label}: allocated {allocated:.2f} GB | reserved {reserved:.2f} GB")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)
vram("After 4-bit load")
print("EOS token:", repr(tokenizer.eos_token))



## 5. Baseline inference (before fine-tuning)

The prompt asks the model to think inside `<think>` tags, matching DeepSeek-R1 style. The baseline answer is usually verbose; fine-tuning tightens the format.


In [ ]:
PROMPT_STYLE = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
{}

### Response:
<think>{}
"""

TRAIN_PROMPT_STYLE = """Below is an instruction that describes a task, paired with an input that provides further context.
Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning.
Please answer the following medical question.

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

BASELINE_QUESTION = (
    "A 61-year-old woman with a long history of involuntary urine loss during activities "
    "like coughing or sneezing but no leakage at night undergoes a gynecological exam and "
    "Q-tip test. Based on these findings, what would cystometry most likely reveal about "
    "her residual volume and detrusor contractions?"
)

NEW_QUESTIONS = [
    BASELINE_QUESTION,
    "A 59-year-old man presents with fevers, chills, and a new murmur after dental work. "
    "What is the most likely diagnosis and the first key investigation?",
    "A 45-year-old man with a 10-year history of alcohol abstinence presents with confusion "
    "and asterixis. Which laboratory pattern and initial management are most appropriate?",
]


def generate_answer(question: str, max_new_tokens: int = 512) -> str:
    FastLanguageModel.for_inference(model)
    inputs = tokenizer([PROMPT_STYLE.format(question, "")], return_tensors="pt").to(model.device)
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        use_cache=True,
    )
    decoded = tokenizer.batch_decode(outputs)[0]
    return decoded.split("### Response:")[-1]


print("===== BEFORE FINE-TUNE =====")
print(generate_answer(BASELINE_QUESTION, max_new_tokens=400))
vram("After baseline generate")



## 6. Medical dataset + tokenization

`FreedomIntelligence/medical-o1-reasoning-SFT` is a clinical Q&A set with a **Question**, a **Complex_CoT** reasoning trace, and a **Response**. We keep the first `N_SAMPLES` English rows and fold them into one `text` field ended by the tokenizer EOS token.


In [ ]:
from datasets import load_dataset

EOS_TOKEN = tokenizer.eos_token


def formatting_prompts_func(examples):
    texts = []
    for question, cot, answer in zip(
        examples["Question"], examples["Complex_CoT"], examples["Response"]
    ):
        texts.append(TRAIN_PROMPT_STYLE.format(question, cot, answer) + EOS_TOKEN)
    return {"text": texts}


dataset = load_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    "en",
    split=f"train[0:{N_SAMPLES}]",
)
print(dataset)
print("Columns:", dataset.column_names)
print("\n--- raw row 0 (truncated) ---")
print("Q:", dataset[0]["Question"][:280], "...")

dataset = dataset.map(formatting_prompts_func, batched=True, desc="Format SFT texts")
print("\n--- formatted text 0 (first 900 chars) ---")
print(dataset["text"][0][:900])
print("...")
print("Avg chars in text field:", sum(len(t) for t in dataset["text"]) / len(dataset))



## 7. LoRA adapter setup (QLoRA)

Only the listed projection layers receive trainable rank-`r` matrices. Base 4-bit weights stay frozen. Unsloth gradient checkpointing cuts activation memory.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()
vram("After LoRA wrap")



## 8. Epoch-based SFT training

`SFTTrainer` + `SFTConfig` (TRL 0.22). Effective batch size = `PER_DEVICE_BATCH * GRAD_ACCUM` (default 8). `MAX_STEPS=60` keeps a T4 run in a reasonable window; `NUM_TRAIN_EPOCHS=1` is the epoch setting requested by the brief.


In [ ]:
from trl import SFTTrainer, SFTConfig

try:
    from unsloth import is_bfloat16_supported
    use_bf16 = is_bfloat16_supported()
except Exception:
    use_bf16 = torch.cuda.is_bf16_supported()

sft_kwargs = dict(
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=5,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=not use_bf16,
    bf16=use_bf16,
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)
if MAX_STEPS and MAX_STEPS > 0:
    sft_kwargs["max_steps"] = MAX_STEPS

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(**sft_kwargs),
)

vram("Before trainer.train()")
trainer_stats = trainer.train()
vram("After trainer.train()")
print(trainer_stats)



## 9. Save the fine-tuned adapter

Only the LoRA weights + tokenizer are written (a few hundred MB), not a full merged 8B dump.


In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to", ADAPTER_DIR.resolve())
print("Files:")
for path in sorted(ADAPTER_DIR.iterdir()):
    size_mb = path.stat().st_size / 1024**2
    print(f"  {path.name:40s} {size_mb:8.2f} MB")

# Optional: merge 16-bit weights (needs extra RAM). Leave False on a free T4.
if False:
    model.save_pretrained_merged("medical_merged_16bit", tokenizer, save_method="merged_16bit")



## 10. Test on new medical queries

Same cystometry item as the baseline, plus two held-out style questions. Compare length and structure of the `<think>` block to section 5.


In [ ]:
print("===== AFTER FINE-TUNE =====\n")
for i, question in enumerate(NEW_QUESTIONS, start=1):
    print(f"--- Query {i} ---")
    print("Q:", question)
    print(generate_answer(question, max_new_tokens=500))
    print()
vram("After post-train generation")



## 11. Submission checklist

- [ ] T4 GPU runtime
- [ ] 4-bit load VRAM printed
- [ ] Dataset formatted with CoT + EOS
- [ ] LoRA trainable-parameter count printed
- [ ] `trainer.train()` finished (loss logged)
- [ ] `medical_qlora_adapter/` saved
- [ ] Post-train answers on the three queries captured

Reload later with:

```python
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="medical_qlora_adapter",
    max_seq_length=2048,
    load_in_4bit=True,
)
```
